# 3 — Constituer le corpus de questions-réponses

Sections 3.1 à 3.4 du livret. Sorties : `data/corpus.csv`, `data/passages.jsonl`,
`data/questions_dev.jsonl`, `data/questions_test.jsonl`.

In [1]:
import sys, json, csv, hashlib, random
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.qa_corpus import ReglagePDF, lire_pdf, lire_html, recoller_coupures, decouper, resoudre_preuves, TAILLE_MAX

DATA = RACINE / "data"
SOURCES = DATA / "sources_qa"
RES = RACINE / "resultats" / "qa"
RES.mkdir(parents=True, exist_ok=True)

## 3.1 Définir le thème et le public

« Cet outil aide **les patients traités par chimiothérapie et leurs proches** à retrouver **les effets
indésirables possibles et leur prise en charge** dans **les documents d'information de l'Institut national
du cancer**. » (fiche de cadrage : `rapport/fiche_cadrage.md`)

## 3.2 Préparer les documents et les passages

1. Documents cohérents par thème, langue et public ; commencer par trois fichiers au texte sélectionnable.
2. Catalogue : identifiant, titre, organisme, URL, date ou version, date de récupération ; conditions
   d'utilisation ; suppression des doublons.
3. Extraction du texte ; comparaison d'un extrait à sa source ; retrait des en-têtes et de la navigation.
4. Découpage en passages selon paragraphes et sections : identifiant, texte, document, localisation source.
5. Contrôle de dix passages, dont le plus court et le plus long : ordre du texte, nombres, unités, négations.


État au 23/10/2026 : 10 documents INCa (2 PDF, 8 pages web ; DOC004 à DOC010 récupérés via la Wayback Machine,
cancer.fr étant en maintenance), découpage complet en passages.

### Catalogue des documents

In [2]:
# Le catalogue data/corpus.csv est rempli à la main (URL, version, date de récupération, conditions d'accès).
# On vérifie ici : un fichier par document, aucun doublon d'identifiant, d'URL ou de contenu.
catalogue = pd.read_csv(DATA / "corpus.csv", dtype=str)
fichiers = {f.name[:6]: f for f in sorted(SOURCES.iterdir()) if f.name.startswith("DOC")}

assert catalogue.document_id.is_unique and catalogue.url.is_unique
assert set(catalogue.document_id) == set(fichiers), "catalogue et dossier sources_qa/ ne correspondent pas"
catalogue["fichier"] = catalogue.document_id.map(lambda d: fichiers[d].name)
catalogue["sha256"] = catalogue.document_id.map(lambda d: hashlib.sha256(fichiers[d].read_bytes()).hexdigest()[:12])
assert catalogue.sha256.is_unique, "deux fichiers identiques"
print(len(catalogue), "documents —", catalogue.organisme.unique())
catalogue[["document_id", "titre", "version_ou_date", "date_recuperation", "fichier", "sha256"]]

10 documents — <ArrowStringArray>
['Institut national du cancer (INCa)']
Length: 1, dtype: str


,document_id,titre,version_ou_date,date_recuperation,fichier,sha256
0,DOC001,"Comprendre la chimiothérapie (guide patients, ...",2011-09 (PDF mis en ligne le 2018-10-15),2026-10-07,DOC001_comprendre_la_chimiotherapie.pdf,1055fe1a94cf
1,DOC002,Cancers et soins de support (fiche Cancer info),2024-11,2026-10-07,DOC002_cancers_et_soins_de_support.pdf,c5bc1de9b7fb
2,DOC003,Les effets indésirables généraux de la chimiot...,2019-04-28,2026-10-07,DOC003_effets_indesirables_generaux.html,febc1625b7bb
3,DOC004,Pourquoi les traitements font tomber les cheve...,2024-07-01,2026-10-23,DOC004_chute_cheveux_poils.html,59c0516f4303
4,DOC005,Douleurs et traitements (page web),2015-05-26,2026-10-23,DOC005_douleurs_et_traitements.html,ea669c349f32
5,DOC006,La prise en charge de la fatigue (page web),2024-03-08,2026-10-23,DOC006_prise_en_charge_fatigue.html,a3c9b7f5a14b
6,DOC007,"Nausées, vomissements, diarrhées (page web)",2024-12-01,2026-10-23,DOC007_nausees_vomissements_diarrhees.html,e982537846ea
7,DOC008,Nutrition (page web),2024-12-01,2026-10-23,DOC008_nutrition.html,69e472c8f777
8,DOC009,Problèmes de bouche (page web),2015-04-08,2026-10-23,DOC009_problemes_de_bouche.html,e7ddc5bc7b48
9,DOC010,Problèmes de peau (page web),2024-07-01,2026-10-23,DOC010_problemes_de_peau.html,0db59d46f85a


### Extraction et découpage en passages

**Réglages d'extraction.** Les deux PDF INCa sont des planches de deux pages imprimées en une ou deux colonnes,
avec des encadrés : l'ordre brut du texte n'est pas l'ordre de lecture. `src/qa_corpus.py` trie les blocs par
page imprimée, flux (texte courant, puis encadrés), colonne et position, retire en-têtes, pieds de page, notes de
marge, légendes de schémas, champs et pointillés de formulaires, renvois vers d'autres brochures, recolle les phrases coupées par une colonne ou une page
et les césures. Pages retenues :

- DOC001 (63 p. PDF) : p. 3 (présentation) et 5 à 58 (chapitres, fiches, glossaire) ; exclues : couverture,
  crédits, sommaire (p. 4), page « Notes » (p. 40), relecteurs et catalogue (p. 59-63).
- DOC002 (13 p. PDF) : p. 2 à 9, jusqu'au titre « Sources de référence » (crédits et catalogue exclus).

Pages web (DOC003-DOC010) : chapô et blocs `wysiwyg` ; fil d'Ariane, partage, « À lire aussi », vignettes de
publications, crédits photo et liens seuls retirés. Les titres (`h2 > h3 > h4`, titres du PDF) ne sont pas copiés
dans le texte : ils sont gardés dans `localisation.section`.

**Découpage.** Paragraphes voisins d'une même section regroupés jusqu'à 800 caractères (le TD RAG utilise 650) ;
jamais à cheval sur deux sections ; un paragraphe plus long est coupé aux frontières de phrases.

In [3]:
# Renvois vers d'autres brochures (« Des informations complémentaires sont disponibles dans… ») : placés en
# bas de colonne, ils coupent le texte et ne contiennent pas d'information utile aux réponses.
RENVOIS = r"^(Pour d|D)es informations (complémentaires|détaillées)"

REGLAGES = {
    "DOC001": ReglagePDF(pages=[3] + list(range(5, 40)) + list(range(41, 59)),
                         taille_encadre=10.75, titres_encadres=["À retenir"],
                         exclure=[RENVOIS, r"^(Nom et dosage des médicaments|Nombre et dates des cures|"
                                           r"Nom du chimiothérapeute|Établissement)$",
                                  r"^1er jour 2e jour", r"^Autres symptômes :$"]),
    "DOC002": ReglagePDF(pages=list(range(2, 10)), taille_titre=11.0, taille_chapitre=30.0,
                         taille_encadre=9.5, taille_corps=10.0, marge_gauche=110, marge_droite=730,
                         arret="SOURCES DE RÉFÉRENCE",
                         exclure=[RENVOIS, r"^Cette fiche a été publiée avec le soutien", r"^LE PANIER DE SOINS DE SUPPORT$"]),
}

paragraphes, recollages = {}, {}
for doc, fichier in fichiers.items():
    ps = lire_pdf(fichier, REGLAGES[doc]) if fichier.suffix == ".pdf" else lire_html(fichier)
    recollages[doc] = recoller_coupures(ps)
    paragraphes[doc] = ps

passages = [p for doc, ps in paragraphes.items() for p in decouper(doc, ps, TAILLE_MAX)]
assert len({p["passage_id"] for p in passages}) == len(passages)
with open(DATA / "passages.jsonl", "w", encoding="utf-8") as f:
    for p in passages:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

bilan = pd.DataFrame([{
    "document_id": doc,
    "paragraphes": len(paragraphes[doc]),
    "coupures_recollees": recollages[doc],
    "sections": len({p.section for p in paragraphes[doc]}),
    "passages": sum(p["document_id"] == doc for p in passages),
    "car_min": min(len(p["texte"]) for p in passages if p["document_id"] == doc),
    "car_median": int(pd.Series([len(p["texte"]) for p in passages if p["document_id"] == doc]).median()),
    "car_max": max(len(p["texte"]) for p in passages if p["document_id"] == doc),
    "mots": sum(len(p["texte"].split()) for p in passages if p["document_id"] == doc),
} for doc in fichiers])
bilan.to_csv(RES / "bilan_passages.csv", index=False)
print(len(passages), "passages écrits dans data/passages.jsonl")
bilan

413 passages écrits dans data/passages.jsonl


,document_id,paragraphes,coupures_recollees,sections,passages,car_min,car_median,car_max,mots
0,DOC001,552,22,114,260,74,548,795,20722
1,DOC002,57,0,9,42,158,573,781,3493
2,DOC003,27,0,7,12,162,537,697,841
3,DOC004,16,0,10,11,245,443,771,836
4,DOC005,40,0,13,17,138,592,748,1351
5,DOC006,25,0,10,15,94,498,798,1134
6,DOC007,16,0,7,9,282,476,800,689
7,DOC008,54,0,14,32,135,559,800,2677
8,DOC009,11,0,7,7,135,330,670,381
9,DOC010,8,0,4,8,179,348,757,558


### Contrôle de dix passages (dont le plus court et le plus long)

In [4]:
# Dix passages : le plus court, le plus long et huit tirés au hasard (graine 42), à comparer à la source
# (ordre du texte, nombres, unités, négations). Les observations sont consignées dans
# resultats/qa/controle_passages.csv.
par_longueur = sorted(passages, key=lambda p: len(p["texte"]))
autres = [p for p in passages if p not in (par_longueur[0], par_longueur[-1])]
controle = [par_longueur[0], par_longueur[-1]] + random.Random(42).sample(autres, 8)
for p in controle:
    print(f"── {p['passage_id']} ({len(p['texte'])} car.) — {p['localisation']}")
    print(p["texte"], "\n")

── DOC001_P029 (74 car.) — {'unite': 'paragraphes', 'section': '3. Le déroulement d’une chimiothérapie', 'paragraphes': [56], 'pages_imprimees': '18'}
La durée d’une chimiothérapie métastatique varie de 3 mois à plus d’un an. 

── DOC008_P001 (800 car.) — {'unite': 'paragraphes', 'section': 'Introduction', 'paragraphes': [1, 2]}
Le cancer, ses traitements et leurs effets indésirables peuvent avoir des conséquences sur votre alimentation. Bien se nourrir, surveiller son poids et veiller à son équilibre nutritionnel est essentiel pour préserver votre masse musculaire, soutenir vos défenses et prévenir la dénutrition. Un accompagnement personnalisé et des conseils pratiques peuvent vous aider à adapter votre alimentation tout au long du parcours de soins. Pour fonctionner, l’organisme a besoin d’énergie et de divers éléments nutritifs apportés par les aliments. Le cancer, ses traitements et leurs effets indésirables peuvent avoir des conséquences sur votre alimentation, et plus particuliè

In [5]:
observations = pd.read_csv(RES / "controle_passages.csv")
assert list(observations.passage_id) == [p["passage_id"] for p in controle], "relancer le contrôle : passages changés"
observations

,passage_id,choix,ordre_du_texte_conforme,nombres_unites_conformes,negations_conservees,commentaire
0,DOC001_P029,le plus court (74 car.),oui,"oui (3 mois, un an)",—,Phrase complète. Passage court : reste de la s...
1,DOC008_P001,le plus long (800 car.),oui,—,—,Chapô + premier paragraphe. La phrase « Le can...
2,DOC005_P004,tirage,oui,—,oui (« avant que la douleur n’empire »),Conforme. Le renvoi final « Pour en savoir plu...
3,DOC001_P059,tirage,oui,—,—,Conforme ; passage d’un seul paragraphe (sous-...
4,DOC001_P013,tirage,oui,—,oui (« n’ont pas pu identifier »),"Conforme, à cheval sur les p. 11-12. Commence ..."
5,DOC008_P016,tirage,oui,—,—,Conforme. Les puces de la liste n’ont pas de p...
6,DOC001_P142,tirage,oui,oui (deux kilos),oui (« ne sont pas eux-mêmes responsables »),Conforme ; même texte que DOC003_P008 (preuve ...
7,DOC001_P127,tirage,oui,—,—,Liste à puces recollée sur une colonne ; ordre...
8,DOC001_P116,tirage,oui,—,"oui (« Il ne faut pas éviter de sortir », doub...",Commence par une puce : la liste « Il est déco...
9,DOC001_P073,tirage,oui,—,—,Conforme.


In [6]:
# Contrôle automatique complémentaire : chaque phrase de chaque passage est recherchée dans le texte brut de sa
# source (PyMuPDF dans l'ordre du fichier, sans réordonnancement ; texte complet de la page HTML), à la ponctuation
# et aux césures près. Une phrase non retrouvée signale un ordre de lecture à vérifier à la main.
import re, pymupdf
from bs4 import BeautifulSoup

def cle(t):
    t = t.replace("’", "'").replace("\u202f", " ").replace("\xa0", " ").replace("\u00ad", "").lower()
    t = re.sub(r"[-‑]\s*\n\s*", "", t)
    return re.sub(r"[^\w']+", " ", re.sub(r"[●•▪]", "", t)).strip()

brut = {}
for doc, fichier in fichiers.items():
    if fichier.suffix == ".pdf":
        with pymupdf.open(fichier) as pdf:
            brut[doc] = cle("\n".join(page.get_text("text") for page in pdf))
    else:
        brut[doc] = cle(BeautifulSoup(fichier.read_text(encoding="utf-8"), "html.parser").get_text(" "))

def retrouvee(phrase, doc):
    c = cle(phrase)
    return c in brut[doc] or c.replace("-", "") in brut[doc].replace("-", "") or c.replace(" ", "") in brut[doc].replace(" ", "")

absentes = [(p["passage_id"], ph) for p in passages
            for ph in re.split(r"(?<=[.!?;:])\s+|\s*•\s*", p["texte"]) if len(ph) > 15 and not retrouvee(ph, p["document_id"])]
n_phrases = sum(len([ph for ph in re.split(r"(?<=[.!?;:])\s+|\s*•\s*", p["texte"]) if len(ph) > 15]) for p in passages)
print(f"{n_phrases} phrases, {len(absentes)} non retrouvées telles quelles dans le texte brut "
      f"({len(absentes) / n_phrases:.1%}), par document :", pd.Series([a[0][:6] for a in absentes]).value_counts().to_dict())
pd.DataFrame(absentes[:10], columns=["passage_id", "phrase"])

2049 phrases, 37 non retrouvées telles quelles dans le texte brut (1.8%), par document : {'DOC001': 34, 'DOC002': 3}


,passage_id,phrase
0,DOC001_P007,L’hormonothérapie* est un traitement qui empêc...
1,DOC001_P027,Occasionnellement (obligation sociale ou famil...
2,DOC001_P037,La façon dont les médicaments d’une polychimio...
3,DOC001_P042,"En outre, des réactions locales au niveau de l..."
4,DOC001_P047,La chimiothérapie est également injectable dan...
5,DOC001_P053,Leur préparation est donc soumise à des procéd...
6,DOC001_P060,Lorsque l’administration des médicaments est t...
7,DOC001_P063,Cathéter simple ou cathéter veineux central Le...
8,DOC001_P070,Il est glissé sous la peau jusqu’à une grosse ...
9,DOC001_P075,Il est nécessaire de signaler rapidement au mé...


## 3.3 Rédiger et répartir les questions

- DEV : 20 questions ; 16 avec réponse, dont au moins 4 à deux passages complémentaires, et 4 sans réponse.
- TEST : 10 questions ; 8 avec réponse, dont au moins 2 à deux passages complémentaires, et 2 sans réponse.
- Reformulations d'une même question dans le même jeu (`family_id`) ; relecture par l'autre membre ;
  désaccords consignés dans un court guide de rédaction (`rapport/guide_redaction_qa.md`).

Premier contrôle (jalon E) : 3 documents et 5 questions annotées — `rapport/cadrage_qa.md`, Q001 à Q005.

In [7]:
# Les preuves sont annotées par un extrait verbatim du document (champ preuves_extraits) : leurs identifiants
# sont recalculés ici à partir du découpage courant, sans modifier les questions ni les éléments attendus.
def mettre_a_jour(chemin):
    if not chemin.exists() or not chemin.read_text(encoding="utf-8").strip():
        return []
    questions = [resoudre_preuves(json.loads(l), passages) for l in chemin.read_text(encoding="utf-8").splitlines() if l.strip()]
    with open(chemin, "w", encoding="utf-8") as f:
        for q in questions:
            f.write(json.dumps(q, ensure_ascii=False) + "\n")
    return questions

dev = mettre_a_jour(DATA / "questions_dev.jsonl")
test = mettre_a_jour(DATA / "questions_test.jsonl")
pd.DataFrame([{k: q[k] for k in ("question_id", "answerable", "evidence_by_element", "equivalents_by_element")}
              for q in dev + test])

,question_id,answerable,evidence_by_element,equivalents_by_element
0,Q001,True,"[[DOC001_P105], [DOC001_P105], [DOC004_P002]]","[[DOC004_P003], [], [DOC004_P009]]"
1,Q002,True,"[[DOC001_P084], [DOC001_P085]]","[[DOC007_P004], [DOC007_P004]]"
2,Q003,True,"[[DOC003_P008], [DOC003_P008], [DOC002_P003]]","[[DOC001_P142], [DOC001_P142], []]"
3,Q004,True,"[[DOC002_P010], [DOC002_P010], [DOC002_P009]]","[[], [], []]"
4,Q005,False,[],[]
5,Q006,True,"[[DOC009_P003], [DOC009_P003], [DOC009_P003]]","[[DOC001_P097], [DOC001_P098], [DOC001_P098]]"
6,Q007,True,"[[DOC008_P014], [DOC008_P014], [DOC008_P014]]","[[], [], []]"
7,Q008,True,"[[DOC008_P014], [DOC008_P014], [DOC008_P014]]","[[], [], []]"
8,Q009,True,"[[DOC001_P124], [DOC001_P124], [DOC001_P124], ...","[[], [], [], []]"
9,Q010,True,"[[DOC007_P008], [DOC007_P009], [DOC007_P009]]","[[DOC001_P093], [DOC001_P093], [DOC001_P092]]"


## 3.4 Mettre au point sur DEV, puis évaluer sur TEST

DEV sert seul aux réglages (découpage, recherche, prompts). TEST reste de côté jusqu'au gel du 06/11/2026,
puis Q0/Q1/Q2 sont exécutés une seule fois ; les questions TEST ne servent pas d'exemples dans les prompts.

In [8]:
ids_passages = {p["passage_id"] for p in passages}
toutes = dev + test
assert len({q["question_id"] for q in toutes}) == len(toutes), "identifiants de questions en double"
for q in toutes:
    assert q["partition"] in ("dev", "test")
    if q["answerable"]:
        assert q["reference"] and q["elements"] and len(q["evidence_by_element"]) == len(q["elements"]), q["question_id"]
        assert all(ids for ids in q["evidence_by_element"]), q["question_id"]
        assert set(q["evidence_ids"]) <= ids_passages, q["question_id"]
    else:
        assert not q["evidence_ids"] and q.get("justification_absence"), q["question_id"]
familles_partagees = {q["family_id"] for q in dev} & {q["family_id"] for q in test}
assert not familles_partagees, f"familles partagées entre DEV et TEST : {familles_partagees}"

def repartition(qs, nom, cible):
    rep = [q for q in qs if q["answerable"]]
    deux = [q for q in rep if len(q["evidence_ids"]) >= 2]
    print(f"{nom} : {len(qs)}/{cible[0]} questions ; {len(rep)}/{cible[1]} avec réponse, dont {len(deux)} à "
          f"deux passages ou plus (≥ {cible[2]}) ; {len(qs) - len(rep)}/{cible[0] - cible[1]} sans réponse")

repartition(dev, "DEV", (20, 16, 4))
repartition(test, "TEST", (10, 8, 2))

DEV : 20/20 questions ; 16/16 avec réponse, dont 7 à deux passages ou plus (≥ 4) ; 4/4 sans réponse
TEST : 10/10 questions ; 8/8 avec réponse, dont 6 à deux passages ou plus (≥ 2) ; 2/2 sans réponse


In [9]:
# Guide de rédaction (rapport/guide_redaction_qa.md), règles 1.1 et 2.1 : questions et références reformulées.
# Plus long segment de mots recopié tel quel depuis les preuves (les nombres et unités peuvent l'allonger).
import re
mots = lambda t: re.findall(r"\w+", t.lower().replace("’", "'"))
textes = {p["passage_id"]: " ".join(mots(p["texte"])) for p in passages}

def segment_copie(texte, ids):
    m, source, best = mots(texte), " ".join(textes[i] for i in ids), 0
    for i in range(len(m)):
        for j in range(i + best + 1, len(m) + 1):
            if " ".join(m[i:j]) not in source:
                break
            best = j - i
    return best

copie = pd.DataFrame([{"question_id": q["question_id"],
                       "mots_reference": len(mots(q["reference"])),
                       "copie_reference": segment_copie(q["reference"], q["evidence_ids"]),
                       "copie_question": segment_copie(q["question"], q["evidence_ids"])}
                      for q in toutes if q["answerable"]])
print("À reformuler (segment recopié > 10 mots) :",
      list(copie.query("copie_reference > 10 or copie_question > 10").question_id) or "aucune")
copie

À reformuler (segment recopié > 10 mots) : aucune


,question_id,mots_reference,copie_reference,copie_question
0,Q001,55,9,4
1,Q002,18,3,5
2,Q003,44,4,4
3,Q004,28,5,3
4,Q006,32,9,1
5,Q007,38,5,2
6,Q008,45,5,4
7,Q009,38,5,1
8,Q010,44,6,4
9,Q011,20,5,2
